In [25]:
import os 
import mne 
import torch 
import torch.nn as nn 
import torch.nn.functional as F 
import torch.optim as optim 
import numpy as np 
from tqdm import tqdm
import pandas as pd 
import matplotlib.pyplot as plt 
import logging 
from torch.utils.data import Dataset, DataLoader 
from sklearn.preprocessing import StandardScaler 
from sklearn.model_selection import train_test_split 
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
import warnings 
from sklearn.metrics import ConfusionMatrixDisplay 
import seaborn as sns 

# 配置日志 
logging.basicConfig( 
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s', 
    handlers=[ logging.StreamHandler(), logging.FileHandler("eeg_model_training.log") ] 
    ) 
logger = logging.getLogger(__name__) 
warnings.filterwarnings('ignore') 

class AvgMaxPool1d(nn.Module):
    def forward(self, x):
        avg = F.adaptive_avg_pool1d(x, 1)
        mx = F.adaptive_max_pool1d(x, 1)
        return torch.cat([avg, mx], dim=1)

# 配置类 
class EEGConfig: 
    def __init__(self, data_dir, test_patients): 
        self.data_dir = data_dir 
        self.test_patients = test_patients
        self.sample_rate = 256 # CHB-MIT数据库采样率 
        self.segment_duration = 2.0 # 每个分析片段的时长(秒)
        self.segment_length = int(self.sample_rate * self.segment_duration) 
        self.batch_size = 32
        self.num_classes = 2 # 0=正常, 1=癫痫 
        self.test_size = 0.1
        self.random_state = 42 
        self.num_epochs = 60 
        self.learning_rate = 0.0005
        self.weight_decay = 1e-6

In [26]:
# ══════════════════════════════════════════════════════════════════
#  TCN 特征提取器
# ══════════════════════════════════════════════════════════════════
class TCNResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, stride, dilation, padding):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size,
                               stride=stride, padding=padding, dilation=dilation)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.drop  = nn.Dropout(0.3)

        p2 = (kernel_size // 2) * dilation
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size,
                               stride=1, padding=p2, dilation=dilation)
        self.bn2   = nn.BatchNorm1d(out_ch)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, 1, stride=stride),
                nn.BatchNorm1d(out_ch)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        sc  = self.shortcut(x)

        # 对齐时序维度
        if sc.shape[-1] != out.shape[-1]:
            diff = sc.shape[-1] - out.shape[-1]
            if diff > 0:
                sc  = sc[..., :-diff]
            else:
                out = out[..., :sc.shape[-1]]

        return self.relu(out + sc)


class EnhancedResTCN(nn.Module):
    def __init__(self, fs: int = 256, feature_dim: int = 128):
        super().__init__()
        k1 = int(50 * (fs / 100))
        s1 = int(5  * (fs / 100))

        self.layer1 = TCNResidualBlock(1,   64,  k1, stride=s1, dilation=1, padding=k1 // 2)
        self.layer2 = TCNResidualBlock(64,  128, 5,  stride=2,  dilation=2, padding=4)
        self.layer3 = TCNResidualBlock(128, 256, 3,  stride=2,  dilation=4, padding=4)

        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc   = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, feature_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2)
        )

    def forward(self, x):
        return self.fc(self.pool(self.layer3(self.layer2(self.layer1(x)))))


class FeatureExtractor(nn.Module):
    """对每个通道独立提取 TCN 特征"""

    def __init__(self, num_channels: int, fs: int = 256, feature_dim: int = 128):
        super().__init__()
        self.tcn = EnhancedResTCN(fs=fs, feature_dim=feature_dim)

    def forward(self, x):
        # x: (B, C, T)
        B, C, T = x.shape
        out = self.tcn(x.reshape(B * C, 1, T))   # (B*C, feature_dim)
        return out.reshape(B, C, -1)              # (B, C, feature_dim)


# ══════════════════════════════════════════════════════════════════
#  先验邻接矩阵（空间距离 + PLV 融合）
# ══════════════════════════════════════════════════════════════════
class PriorMatrixBuilder(nn.Module):
    """
    构建可学习的先验邻接矩阵 A_prior：
      A_prior = σ( β * (w₀·A_geo + w₁·A_plv_mean) ⊙ softplus(E) − τ )
    """

    _EEG_COORDS = {
        'FP1':  (-0.30,  0.95,  0.10), 'FP2':  ( 0.30,  0.95,  0.10),
        'F7':   (-0.71,  0.50,  0.50), 'F8':   ( 0.71,  0.50,  0.50),
        'F3':   (-0.45,  0.60,  0.65), 'F4':   ( 0.45,  0.60,  0.65),
        'FZ':   ( 0.00,  0.60,  0.80),
        'T3':   (-0.95,  0.00,  0.30), 'T4':   ( 0.95,  0.00,  0.30),
        'T7':   (-0.95,  0.00,  0.30), 'T8':   ( 0.95,  0.00,  0.30),
        'C3':   (-0.71,  0.00,  0.71), 'C4':   ( 0.71,  0.00,  0.71),
        'CZ':   ( 0.00,  0.00,  1.00),
        'T5':   (-0.71, -0.50,  0.50), 'T6':   ( 0.71, -0.50,  0.50),
        'P7':   (-0.71, -0.50,  0.50), 'P8':   ( 0.71, -0.50,  0.50),
        'P3':   (-0.45, -0.60,  0.65), 'P4':   ( 0.45, -0.60,  0.65),
        'PZ':   ( 0.00, -0.60,  0.80),
        'O1':   (-0.30, -0.95,  0.10), 'O2':   ( 0.30, -0.95,  0.10),
        'OZ':   ( 0.00, -1.00,  0.00),
        'FT9':  (-1.00,  0.25,  0.10), 'FT10': ( 1.00,  0.25,  0.10),
        'A1':   (-1.00, -0.10, -0.10), 'A2':   ( 1.00, -0.10, -0.10),
    }

    def __init__(self, channel_names, sigma=1.0, beta=10.0, tau=0.5):
        super().__init__()
        self.C    = len(channel_names)
        self.beta = beta
        self.tau  = tau

        pos  = torch.tensor([self._resolve(n) for n in channel_names], dtype=torch.float32)
        dist = torch.cdist(pos, pos, p=2)
        A_geo = torch.exp(-(dist ** 2) / (sigma ** 2))
        self.register_buffer('A_geo', A_geo)

        self.weight_fusion = nn.Parameter(torch.tensor([0.5, 0.5]))
        self.E             = nn.Parameter(torch.randn(self.C, self.C) * 0.01)

    def _resolve(self, name: str):
        u = name.upper().replace(' ', '')
        if u in self._EEG_COORDS:
            return list(self._EEG_COORDS[u])
        clean = u.replace('EEG', '').strip().lstrip('-')
        if '-' in clean:
            parts = clean.split('-')
            coords = []
            for p in parts[:2]:
                c = self._EEG_COORDS.get(p)
                if c is None:
                    for k, v in self._EEG_COORDS.items():
                        if p.startswith(k) or k.startswith(p):
                            c = v; break
                if c is not None:
                    coords.append(c)
            if len(coords) == 2:
                return [(coords[0][i] + coords[1][i]) / 2 for i in range(3)]
            if len(coords) == 1:
                return list(coords[0])
        return [0.0, 0.0, 0.0]

    def forward(self, A_plv: torch.Tensor) -> torch.Tensor:
        # A_plv: (B, C, C)
        A_plv_mean = A_plv.mean(dim=0)                           # (C, C)
        w   = F.softmax(self.weight_fusion, dim=0)
        A0  = w[0] * self.A_geo + w[1] * A_plv_mean
        Asp = F.softplus(self.E)
        A_init = A0 * Asp
        A_init = (A_init + A_init.T) / 2                        # 对称化
        mask   = 1.0 - torch.eye(self.C, device=A_init.device)
        A_init = A_init * mask                                   # 去自环
        return torch.sigmoid(self.beta * (A_init - self.tau))   # (C, C)


# ══════════════════════════════════════════════════════════════════
#  图注意力层（带先验偏置）
# ══════════════════════════════════════════════════════════════════
class GraphAttentionLayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, c_dim: int = 64, dropout: float = 0.3):
        super().__init__()
        self.W       = nn.Linear(in_dim, out_dim, bias=False)
        self.a       = nn.Parameter(torch.empty(2 * out_dim + c_dim, 1))
        self.dropout = nn.Dropout(dropout)
        self.c_dim   = c_dim
        nn.init.xavier_uniform_(self.a.data, gain=1.414)

    def forward(self, h: torch.Tensor, adj=None, c=None,
                prior_adj=None, prior_lambda: float = 1.0, eps: float = 1e-8):
        Wh = self.W(h)                            # (B, N, out_dim)
        B, N, D = Wh.shape

        Zi = Wh.unsqueeze(2).expand(-1, N, N, -1)
        Zj = Wh.unsqueeze(1).expand(-1, N, N, -1)

        if c is None:
            c = torch.zeros(B, 1, 1, self.c_dim, device=Wh.device)
        else:
            if c.dim() == 2:                      # (B, c_dim)
                c = c.unsqueeze(1).unsqueeze(1)   # (B, 1, 1, c_dim)
            c = c.expand(B, N, N, self.c_dim)

        a_input = torch.cat([Zi, Zj, c], dim=-1)
        e = F.leaky_relu(torch.matmul(a_input, self.a).squeeze(-1), negative_slope=0.2)

        if prior_adj is not None:
            if prior_adj.dim() == 2:
                prior_adj = prior_adj.unsqueeze(0).expand(B, -1, -1)
            e = e + prior_lambda * torch.log(prior_adj.clamp(min=eps))

        if adj is not None:
            if adj.dim() == 2:
                adj = adj.unsqueeze(0).expand(B, -1, -1)
            e = e.masked_fill(adj == 0, -1e9)

        alpha = self.dropout(F.softmax(e, dim=-1))
        return F.elu(torch.bmm(alpha, Wh))        # (B, N, out_dim)


# ══════════════════════════════════════════════════════════════════
#  TAGAT（去掉 TimeDelayEncoding / seq_len，用通道均值作 context）
# ══════════════════════════════════════════════════════════════════
class TAGAT(nn.Module):
    """
    时间–感知图注意力（单片段版）。
    原 seq_len 相关的时间延迟编码已移除；
    改用当前片段所有通道特征的均值经 MLP 投影作 context，
    保留对通道间拓扑关系的建模能力。
    """

    def __init__(self, feature_dim: int, hid_dim: int, c_dim: int = 64):
        super().__init__()
        self.c_proj = nn.Sequential(
            nn.Linear(feature_dim, c_dim),
            nn.ReLU(inplace=True)
        )
        self.gat1 = GraphAttentionLayer(feature_dim, hid_dim, c_dim=c_dim, dropout=0.3)
        self.gat2 = GraphAttentionLayer(hid_dim,     hid_dim, c_dim=c_dim, dropout=0.3)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, feature_dim)
        c   = self.c_proj(x.mean(dim=1))              # (B, c_dim)
        N   = x.size(1)
        adj = torch.ones(N, N, device=x.device)
        x   = self.gat1(x, adj=adj, c=c)
        x   = self.gat2(x, adj=adj, c=c)
        return x                                       # (B, C, hid_dim)


# ══════════════════════════════════════════════════════════════════
#  SCGAT（空间–条件图注意力，引入先验邻接矩阵）
# ══════════════════════════════════════════════════════════════════
class SCGAT(nn.Module):
    def __init__(self, feature_dim: int, hid_dim: int,
                 num_classes: int = 2, c_dim: int = 64):
        super().__init__()
        self.gat1 = GraphAttentionLayer(feature_dim, hid_dim, c_dim=c_dim, dropout=0.3)
        self.gat2 = GraphAttentionLayer(hid_dim,     hid_dim, c_dim=c_dim, dropout=0.3)
        self.aux_fc    = nn.Linear(hid_dim, num_classes)
        self.cond_mlp  = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, c_dim)
        )

    def forward(self, x: torch.Tensor, adj_prior: torch.Tensor):
        # x: (B, C, feature_dim)  adj_prior: (C, C)
        B, N = x.size(0), x.size(1)
        c    = self.cond_mlp(x.mean(dim=1))            # (B, c_dim)
        adj_full = torch.ones(N, N, device=x.device)

        if adj_prior.dim() == 2:
            adj_prior = adj_prior.unsqueeze(0).expand(B, -1, -1)

        x = self.gat1(x, adj=adj_full, c=c, prior_adj=adj_prior, prior_lambda=1.0)
        x = self.gat2(x, adj=adj_full, c=c, prior_adj=adj_prior, prior_lambda=1.0)

        aux_logits = self.aux_fc(x.mean(dim=1))        # (B, num_classes)
        return x, aux_logits                            # (B, C, hid_dim), (B, num_classes)


# ══════════════════════════════════════════════════════════════════
#  门控融合 & 通道门控
# ══════════════════════════════════════════════════════════════════
class GatedFusion(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.fc = nn.Linear(2 * dim, dim)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        z = torch.sigmoid(self.fc(torch.cat([x1, x2], dim=-1)))
        return z * x1 + (1 - z) * x2


class ChannelGating(nn.Module):
    def __init__(self, num_channels: int, hid_dim: int):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(hid_dim, num_channels),
            nn.Sigmoid()
        )

    def forward(self, fused: torch.Tensor, ch_feats: torch.Tensor) -> torch.Tensor:
        w = self.gate(fused).unsqueeze(-1)             # (B, C, 1)
        return (ch_feats * w).sum(dim=1)               # (B, hid_dim)


# ══════════════════════════════════════════════════════════════════
#  在线 PLV 计算（GPU 友好）
# ══════════════════════════════════════════════════════════════════
def compute_plv_batch(x: torch.Tensor) -> torch.Tensor:
    """
    Phase Locking Value (PLV) 矩阵，在 forward 中在线计算。

    输入：x  (B, C, T)   float32
    输出：PLV (B, C, C)  float32，值域 [0, 1]

    算法：
        X   = rfft(x)                        → (B, C, F) complex
        Xn  = X / |X|                        → 单位复数（仅保留相位）
        PLV = |Xn @ conj(Xn)ᵀ| / F          → 通道间相位同步程度
    """
    X    = torch.fft.rfft(x, dim=-1)                  # (B, C, F) complex
    Xn   = X / X.abs().clamp(min=1e-8)                # 单位复数
    F_   = Xn.shape[-1]
    # bmm: (B, C, F) × (B, F, C) → (B, C, C)
    plv  = torch.bmm(Xn, Xn.conj().transpose(1, 2)).abs() / F_
    return plv                                         # float32, [0, 1]


# ══════════════════════════════════════════════════════════════════
#  主网络（无 BiLSTM，输入 (B, C, L)）
# ══════════════════════════════════════════════════════════════════
class EpilepsyGATNet(nn.Module):
    """
    核心改动：
    · 去掉 context_bilstm，直接对融合后的向量分类
    · 输入从 (B, S, C, L) 变为 (B, C, L)
    · PLV 在 forward 内在线计算，无需外部传入 a_plv
    · TAGAT 去掉 TimeDelayEncoding，改用通道均值 MLP 作 context
    """

    def __init__(
        self,
        channel_names = ["FP1-F7","F7-T7","T7-P7","P7-O1","FP1-F3","F3-C3","C3-P3","P3-O1","FP2-F4","F4-C4","C4-P4","P4-O2","FP2-F8","F8-T8","T8-P8",
                         "P8-O2","FZ-CZ","CZ-PZ","P7-T7","T7-FT9","FT9-FT10","FT10-T8"],
        fs: int          = 256,
        num_classes: int = 2,
        feature_dim: int = 128,
        hid_dim: int     = 256,
    ):
        super().__init__()
        self.C     = len(channel_names)
        self.c_dim = 64

        self.prior_builder  = PriorMatrixBuilder(channel_names)
        self.feat_extractor = FeatureExtractor(self.C, fs, feature_dim)
        self.tagat          = TAGAT(feature_dim, hid_dim, c_dim=self.c_dim)
        self.scgat          = SCGAT(feature_dim, hid_dim, num_classes, c_dim=self.c_dim)

        self.norm_t    = nn.LayerNorm(hid_dim)
        self.norm_s    = nn.LayerNorm(hid_dim)
        self.gate_fuse = GatedFusion(hid_dim)
        self.ch_gate   = ChannelGating(self.C, hid_dim)

        # 直接分类（BiLSTM 已移除）
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(hid_dim, hid_dim // 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hid_dim // 2, num_classes)
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor):
        # x: (B, C, L)
        A_plv   = compute_plv_batch(x)               # (B, C, C)  在线 PLV
        A_prior = self.prior_builder(A_plv)           # (C, C)

        feat    = self.feat_extractor(x)              # (B, C, feature_dim)

        t_out   = self.norm_t(self.tagat(feat))       # (B, C, hid_dim)
        s_out, s_logits = self.scgat(feat, A_prior)   # (B, C, hid_dim), (B, num_cls)
        s_out   = self.norm_s(s_out)

        fused   = self.gate_fuse(t_out.mean(1), s_out.mean(1))   # (B, hid_dim)
        ch_g    = self.ch_gate(fused, s_out)                      # (B, hid_dim)
        fused   = fused + ch_g

        logits  = self.classifier(fused)              # (B, num_classes)
        return logits

In [27]:
def prepare_csv_dataset(config):
    """
    从 CHB-MIT CSV 切片数据集中生成:
        X -> [N, 22, 512]
        y -> [N]
        patient_ids -> [N]

    ⚠️ 标签规则（与你现有代码保持一致）:
        0.csv      -> 非发作 (0)
        其他 csv   -> 发作 (1)
    """

    root_dir = config.data_dir
    segment_length = config.segment_length

    # ===== CHB-MIT EEG 通道数 =====
    num_channels = 22

    all_segments = []
    all_labels = []
    all_patient_ids = []

    for patient in sorted(os.listdir(root_dir)):
        patient_dir = os.path.join(root_dir, patient)
        if not os.path.isdir(patient_dir):
            continue

        for file in sorted(os.listdir(patient_dir)):
            if not file.endswith(".csv"):
                continue

            csv_path = os.path.join(patient_dir, file)

            # 标签规则（不变）
            label = 0 if file == "0.csv" else 1

            try:
                df = pd.read_csv(csv_path, header=None)

                # 全部转成数值，非法值直接丢
                df = df.apply(pd.to_numeric, errors="coerce").dropna()

                # ⭐ 直接取前 22 列（不再检查列数）
                eeg = df.iloc[:, :num_channels].values.astype(np.float32)
                total_samples = len(eeg)

                if total_samples < segment_length:
                    print(f" {file}: 数据 {total_samples} < {segment_length}，跳过")
                    continue

                num_segments = total_samples // segment_length

                for i in range(num_segments):
                    seg = eeg[i*segment_length:(i+1)*segment_length]
                    seg = seg.T  # [29, 512]

                    all_segments.append(seg)
                    all_labels.append(label)
                    all_patient_ids.append(patient)

            except Exception as e:
                print(f"❌ 读取出错 {csv_path}: {e}")
                continue

    if len(all_segments) == 0:
        print("❌ 没有生成任何片段")
        return None, None, None

    X = np.array(all_segments, dtype=np.float32)
    y = np.array(all_labels, dtype=np.int64)
    patient_ids = np.array(all_patient_ids)

    print(f"\n🎉 最终生成 {len(X)} 个片段")
    print(f"📐 片段形状 = {X.shape[1:]}")
    print(f"📌 标签分布: 正常={np.sum(y==0)}, 发作={np.sum(y==1)}")

    return X, y, patient_ids

# 训练函数
def train(model, device, train_loader, optimizer, criterion, epoch): 
    model.train() 
    total_loss = 0

    for batch_idx, (data, target) in enumerate(train_loader): 
        data, target = data.to(device), target.to(device) 
        optimizer.zero_grad() 
        output = model(data) 
        loss = criterion(output, target) 
        loss.backward() 
        optimizer.step() 
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    return avg_loss
    
# 测试函数 
def test(model, device, loader, criterion):
    model.eval()

    y_true, y_pred, y_score = [], [], []
    total_loss = 0.0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            loss = criterion(outputs, y)

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            total_loss += loss.item() * x.size(0)

            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            y_score.extend(probs[:, 1].cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_score)
    prec = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    return avg_loss, acc, auc, prec, recall, f1


# 主函数 
# === 可视化模块1：Confusion Matrix 可视化 === 
def plot_confusion_matrix(y_true, y_pred, config): 
    """绘制混淆矩阵并保存""" 
    cm = confusion_matrix(y_true, y_pred) 
    plt.figure(figsize=(6, 5)) 
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
                xticklabels=["Normal (0)", "Seizure (1)"], 
                yticklabels=["Normal (0)", "Seizure (1)"]) 
    
    plt.title("Confusion Matrix", fontdict={'fontsize': 30}) 
    plt.xlabel(xlabel="Predicted Label", fontdict={'fontsize': 24}) 
    plt.ylabel(ylabel="True Label", fontdict={'fontsize': 24}) 
    plt.tight_layout() 

    save_path = os.path.join(config.data_dir, "confusion_matrix.png") 
    plt.savefig(save_path) 
    logger.info(f"混淆矩阵图已保存至: {save_path}") 
    plt.close() 

# === 训练过程曲线可视化 === 
def plot_training_summary_with_std(metrics, save_path=None):
    """ 
    绘制训练总结图： 
    1. Loss 折线图（train + test） 
    2. AUC 折线图 
    3. 平均 Accuracy / Precision / Recall / F1 柱状图（带标准差误差条） 
    
    参数: 
    metrics: dict 包含 'train_loss', 'test_loss', 'accuracy', 'auc', 'precision', 'recall', 'f1' 
    save_path: str or None, 若提供则保存为文件 
    """ 
    epochs = np.arange(1, len(metrics['train_loss']) + 1) 
    
    # ==== 全局字体设置 ==== 
    fig, axes = plt.subplots(1, 3, figsize=(20, 6)) 
    
    # === 1️⃣ Loss 折线图 === 
    plt.plot(epochs, metrics['train_loss'], 'b-', label='Train Loss') 
    plt.plot(epochs, metrics['test_loss'], 'r-', label='Test Loss') 
    plt.title('Loss', fontsize=24) 
    plt.xlabel('Epoch', fontsize=20) 
    plt.ylabel('Loss', fontsize=20) 
    plt.ylim(0, 1) 
    plt.legend() 
    
    # === 2️⃣ Accuracy 折线图 === 
    plt.subplot(1, 3, 2)
    plt.plot(epochs, metrics['accuracy'], 'm-', linewidth=2) 
    plt.axhline(y=0.99, color='red', linestyle='--', linewidth=2, label='99% Threshold')
    plt.title('Accuracy', fontsize=24) 
    plt.xlabel('Epoch', fontsize=20) 
    plt.ylabel('Accuracy', fontsize=20) 
    plt.grid(True, linestyle='--', alpha=0.5) 
    
    # === 3️⃣ 平均值 + 标准差 柱状图 === 
    ax3 = axes[2] 
    
    # 计算平均值和标准差 
    avg_acc = np.mean(metrics['accuracy']) * 100 
    avg_prec = np.mean(metrics['precision']) * 100 
    avg_recall = np.mean(metrics['recall']) * 100 
    avg_f1 = np.mean(metrics['f1']) * 100 

    std_acc = np.std(metrics['accuracy']) * 100 
    std_prec = np.std(metrics['precision']) * 100 
    std_recall = np.std(metrics['recall']) * 100 
    std_f1 = np.std(metrics['f1']) * 100 

    bars = ['Accuracy', 'Precision', 'Recall', 'F1 Score'] 
    means = [avg_acc, avg_prec, avg_recall, avg_f1] 
    stds = [std_acc, std_prec, std_recall, std_f1] 
    colors = ['#4CAF50', '#2196F3', '#FFC107', '#9C27B0'] 
    
    ax3.bar(bars, means, yerr=stds, color=colors, alpha=0.8, capsize=8) 
    ax3.set_xlabel('Metrics', fontsize=20) 
    ax3.set_ylabel('Score (%)', fontsize=20) 
    ax3.set_ylim(0, 100) 
    plt.tight_layout() 
    
    # === 保存 === 
    if save_path: 
        plt.savefig(save_path, dpi=300, bbox_inches='tight') 
        print(f"✅ 图像已保存到: {save_path}") 

        plt.show() 
        
# === 修改后的主函数 === 
def main():
    data_dir = "epilepsy/project_epilepsy/CHBMITnew5"
    config = EEGConfig(
        data_dir=data_dir,
        test_patients=["chb06", "chb08", "chb10"]  # ⭐ 在这里指定要跑的受试者列表
    )

    # === 1. 加载数据 ===
    logger.info("准备 CSV 数据集...")
    X, y, patient_ids = prepare_csv_dataset(config)

    unique_patients = np.unique(patient_ids)
    logger.info(f"共 {len(unique_patients)} 个病人: {unique_patients}")

    # === 2. 验证列表中的受试者是否都存在 ===
    invalid = [p for p in config.test_patients if p not in unique_patients]
    if invalid:
        raise ValueError(
            f"❌ 以下受试者不在数据集中: {invalid}\n"
            f"   可用受试者: {list(unique_patients)}"
        )

    all_fold_metrics = []

    # === 3. 按列表轮流跑 ===
    for fold, test_pid in enumerate(config.test_patients):
        logger.info(f"\n===== Fold {fold+1}/{len(config.test_patients)} | "
                    f"Test Patient = {test_pid} =====")

        train_idx = patient_ids != test_pid
        test_idx  = patient_ids == test_pid

        X_train, y_train = X[train_idx], y[train_idx]
        X_test,  y_test  = X[test_idx],  y[test_idx]

        X_train_sub, X_val, y_train_sub, y_val = train_test_split(
            X_train,
            y_train,
            test_size=0.1,
            random_state=42,
            stratify=y_train
        )

        logger.info(
            f"Train: {len(y_train_sub)} | "
            f"Val: {len(y_val)} | "
            f"Test: {len(y_test)} | "
            f"Test seizure ratio: {np.mean(y_test):.4f}"
        )

        # === 4. Dataset / DataLoader ===
        train_dataset = torch.utils.data.TensorDataset(
            torch.tensor(X_train_sub, dtype=torch.float32),
            torch.tensor(y_train_sub, dtype=torch.long)
        )
        val_dataset = torch.utils.data.TensorDataset(
            torch.tensor(X_val, dtype=torch.float32),
            torch.tensor(y_val, dtype=torch.long)
        )
        test_dataset = torch.utils.data.TensorDataset(
            torch.tensor(X_test, dtype=torch.float32),
            torch.tensor(y_test, dtype=torch.long)
        )

        train_loader = DataLoader(train_dataset, batch_size=config.batch_size,
                                  shuffle=True, drop_last=True)
        val_loader   = DataLoader(val_dataset,   batch_size=config.batch_size,
                                  shuffle=False)
        test_loader  = DataLoader(test_dataset,  batch_size=config.batch_size,
                                  shuffle=False)

        # === 5. 模型 ===
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        model = nn.Sequential(
            EpilepsyGATNet(),
            nn.LogSoftmax(dim=1)
        ).to(device)

        criterion = nn.NLLLoss()
        optimizer = torch.optim.Adam(model.parameters(),
                                     lr=0.001,
                                     weight_decay=config.weight_decay)

        best_val_acc    = -1.0
        best_epoch      = -1
        best_model_path = f"best_model_test_{test_pid}.pth"

        # === 6. 训练 ===
        for epoch in range(1, config.num_epochs + 1):
            logger.info(f"\n--- Epoch {epoch}/{config.num_epochs} ---")

            train_loss = train(model, device, train_loader,
                               optimizer, criterion, epoch)

            val_loss, val_acc, val_auc, val_prec, val_recall, val_f1 = test(
                model, device, val_loader, criterion
            )

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_epoch   = epoch
                torch.save(model.state_dict(), best_model_path)
                logger.info(f"✅ [BEST UPDATE] Epoch {epoch}: "
                            f"Val Acc = {val_acc:.4f}")

            logger.info(
                f"[Fold {fold+1} | Epoch {epoch:03d}] "
                f"Train Loss = {train_loss:.4f} | "
                f"Val Loss = {val_loss:.4f} | "
                f"Val Acc = {val_acc:.4f}"
            )

        logger.info(
            f"🏁 [Fold {fold+1}] Training finished | "
            f"Best Epoch = {best_epoch} | "
            f"Best Val Acc = {best_val_acc:.4f}"
        )

        # === 7. 测试 ===
        model.load_state_dict(torch.load(best_model_path))
        model.eval()

        test_loss, acc, auc, prec, recall, f1 = test(
            model, device, test_loader, criterion
        )

        logger.info(
            f"\n===== RESULT | Test Patient = {test_pid} =====\n"
            f"Best Val Acc = {best_val_acc:.4f} (Epoch {best_epoch})\n"
            f"Test Acc = {acc:.4f} | AUC = {auc:.4f} | "
            f"P = {prec:.4f} | R = {recall:.4f} | F1 = {f1:.4f}"
        )

        all_fold_metrics.append({
            "patient": test_pid,
            "acc": acc, "auc": auc,
            "prec": prec, "recall": recall, "f1": f1
        })

    # === 8. 汇总所有受试者结果 ===
    logger.info("\n===== Overall Results =====")
    logger.info(f"{'Patient':<10} {'Acc':>8} {'AUC':>8} "
                f"{'Prec':>8} {'Recall':>8} {'F1':>8}")
    logger.info("-" * 52)

    for m in all_fold_metrics:
        logger.info(
            f"{m['patient']:<10} {m['acc']:>8.4f} {m['auc']:>8.4f} "
            f"{m['prec']:>8.4f} {m['recall']:>8.4f} {m['f1']:>8.4f}"
        )

    # 均值 ± 标准差
    metrics_arr = np.array([[m['acc'], m['auc'], m['prec'],
                              m['recall'], m['f1']] for m in all_fold_metrics])
    mean = metrics_arr.mean(axis=0)
    std  = metrics_arr.std(axis=0)

    logger.info("-" * 52)
    logger.info(
        f"{'Mean':<10} {mean[0]:>8.4f} {mean[1]:>8.4f} "
        f"{mean[2]:>8.4f} {mean[3]:>8.4f} {mean[4]:>8.4f}"
    )
    logger.info(
        f"{'Std':<10} {std[0]:>8.4f} {std[1]:>8.4f} "
        f"{std[2]:>8.4f} {std[3]:>8.4f} {std[4]:>8.4f}"
    )


if __name__ == "__main__":
    main()

2026-05-26 06:08:47,030 - INFO - 准备 CSV 数据集...


FileNotFoundError: [Errno 2] No such file or directory: 'epilepsy/project_epilepsy/CHBMITnew5'